# VectorDrugMatch - treats-edge ablation (GCN message passing), Colab runQuestion: does letting **training-positive** `treats` edges take part in message passing help a GCN on the drug-disjoint benchmark, without target leakage? Design, leakage safeguards and caveats: `Docs/TREATS_EDGE_ABLATION_PLAN.md`.Run the cells **top to bottom, one at a time**. The code is pinned to commit `6b6fd6380f` on branch `exp/treats-edge-ablation` (it must be on GitHub first). Nothing here changes the completed results: outputs go to a new folder `VDM_Outputs/treats_edge_ablation_20261111/`.Prerequisite: `MyDrive/VDM_Data/frozen_processed/primekg/` exists (the upload from the main handbook).

## Step 1 - GPURuntime > Change runtime type > T4 GPU.

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
!nvidia-smi | head -15

## Step 2 - Drive, code at the pinned commit, libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf VectorDrugMatch
!git clone https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd VectorDrugMatch
PIN = '6b6fd6380f28ce43cfc948590bb91730c3051b6e'
!git checkout $PIN
!git log -1 --oneline

In [ ]:
%pip install -q rdkit mlflow shap xgboost loguru pyyaml torch-geometric tabulate scikit-learn lxml matplotlib pymupdf pyarrow
import torch, torch_geometric
print('PyTorch', torch.__version__, '| PyG', torch_geometric.__version__, '| CUDA', torch.cuda.is_available())
try:
    import pyg_lib; print('pyg-lib', pyg_lib.__version__)
except Exception as e:
    print('pyg-lib not available (only needed for the RGCN configuration):', e)

## Step 3 - Put the frozen benchmark in place and verify it (stop on a mismatch)

In [ ]:
import hashlib, json, os, shutil
D = '/content/drive/MyDrive/VDM_Data/frozen_processed/primekg'
assert os.path.isdir(D), 'upload MyDrive/VDM_Data/frozen_processed/primekg first (see the main handbook)'
shutil.copytree(D, 'data/processed/primekg', dirs_exist_ok=True)
man = json.load(open('Docs/release/vdm_v1_candidate_manifest.json'))['files']
want = {f['path']: f['sha256'] for f in man if f['path'].startswith('data/processed/primekg/') and f['path'].endswith(('.csv', '.npy', '.npz'))}
bad = [p for p, h in want.items() if hashlib.sha256(open(p, 'rb').read()).hexdigest() != h]
print(len(want), 'files checked;', 'MISMATCH:' if bad else 'FROZEN BENCHMARK VERIFIED', bad)
assert not bad

## Step 4 - Outputs to Drive (a new folder; the completed results are not touched)

In [ ]:
import os
O = '/content/drive/MyDrive/VDM_Outputs/treats_edge_ablation_20261111'
os.makedirs(O, exist_ok=True); os.makedirs('outputs', exist_ok=True)
if not os.path.islink('outputs/treats_edge_ablation_20261111'):
    os.symlink(O, 'outputs/treats_edge_ablation_20261111')
print(os.path.realpath('outputs/treats_edge_ablation_20261111'))

## Step 5 - Graph diagnostics and the automated leakage audit (CPU)Must print `all_pass=True` for the three graph variants. Compare with `reports/treats_edge_ablation_20261111/diagnostics/` (computed locally). Stop on any FAIL.

In [ ]:
!python scripts/graph_diagnostics.py

## Step 6 - Smoke test (every configuration, seed 42, 3 epochs; written to `smoke/`, never mixed with results)Checks that every configuration runs and passes its leakage audit. Note the seconds it prints: they are the first timing measurement.

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python scripts/run_treats_edge_ablation.py --smoke

## Step 7 - Sensitivity of the masking fraction k (validation ROC-AUC only, seed 42)k is the number of folds per epoch: 1/k of the train positives is hidden from message passing in each step. **k = 4 is fixed a priori for the main runs**; this table only shows sensitivity and is not used to pick the final configuration after seeing test scores.

In [ ]:
!python scripts/run_treats_edge_ablation.py --config-name B_k2 B_k4 B_k8 --seeds 42 --resume
import json, glob
for c in ('B_k2', 'B_k4', 'B_k8'):
    for f in glob.glob(f'outputs/treats_edge_ablation_20261111/{c}/seed42/metrics.json'):
        print(c, 'val ROC-AUC', round(json.load(open(f))['val']['roc_auc'], 4))

## Step 8 - Main runs, five seeds (21, 42, 52, 72, 100). Resumable: re-run this cell after a disconnect (Steps 1-4 first).Order: the control and the main comparison first.

In [ ]:
!python scripts/run_treats_edge_ablation.py --config-name A_orig A_k4 B_k4 --resume

In [ ]:
!python scripts/run_treats_edge_ablation.py --config-name C_k4 A_k4_L3 B_k4_L3 --resume

In [ ]:
# optional relation-aware comparison and the DIAGNOSTIC naive-inclusion run (target leakage by construction; never report B0 as a result)
!python scripts/run_treats_edge_ablation.py --config-name D_rgcn_B B0_naive --resume

## Step 9 - Summarise and saveWrites `seed_wise_metrics.csv`, `five_seed_summary.csv`, `paired_vs_A_orig.csv` and `README_results.md` into the Drive folder. Report these numbers as they are; see the plan for how to interpret them.

In [ ]:
!python scripts/summarize_treats_ablation.py --reference A_orig
!cd outputs && zip -rq /content/drive/MyDrive/VDM_Outputs/treats_edge_ablation_20261111_results_light.zip treats_edge_ablation_20261111 -x '*/model.pth'
print(open('outputs/treats_edge_ablation_20261111/README_results.md').read())